In [1]:
%pip install --upgrade transformers==4.45.2 datasets accelerate trl==0.11.3 wandb evaluate tensorboard -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install weave -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
# ================= IMPORT =================
import os
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt

/home/nam/projects/hoang/Advanced-ASM1/rlhf/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
/home/nam/projects/hoang/Advanced-ASM1/rlhf/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ================= CONFIG =================
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "gpt2"
MAX_LENGTH = 256
MAX_NEW_TOKENS = 50
BATCH_SIZE = 4
EPOCHS = 3

LR_SFT = 1e-5
LR_REWARD = 1e-6
LR_PPO = 1e-6
LR_DPO = 5e-7

PPO_BETA = 0.02
PPO_CLIP = 0.2
DPO_BETA = 0.1

OUT_DIR = "out_manual"
os.makedirs(OUT_DIR, exist_ok=True)

In [5]:
# ================= DATA =================
def load_pref():
    ds = load_dataset("HumanLLMs/Human-Like-DPO-Dataset", split="train")
    ds = ds.train_test_split(test_size=0.2, seed=42)
    return ds["train"], ds["test"]

train_pref, val_pref = load_pref()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

In [6]:
# ================= UTILS =================
def generate(model, input_ids, attention_mask):
    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            top_p=0.9,
            temperature=0.7,
            pad_token_id=tokenizer.pad_token_id
        )
    attn = (out != tokenizer.pad_token_id).long()
    return out, attn

def compute_logprob(model, ids, attention_mask, prompt_len):
    out = model(input_ids=ids, attention_mask=attention_mask)
    logits = out.logits[:, :-1]
    labels = ids[:, 1:]
    mask = attention_mask[:, 1:]

    gen_mask = mask.clone()
    gen_mask[:, :prompt_len-1] = 0

    logp = F.log_softmax(logits, dim=-1)
    token_logp = logp.gather(-1, labels.unsqueeze(-1)).squeeze(-1)

    return (token_logp * gen_mask).sum(-1)

def masked_mean(hidden, mask):
    return (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)

In [7]:
# ================= SFT =================
def sft_collate(batch):
    texts = [x["prompt"] + x["chosen"] for x in batch]
    enc = tokenizer(texts, truncation=True, max_length=MAX_LENGTH,
                    padding=True, return_tensors="pt")
    enc["labels"] = enc["input_ids"].clone()
    return enc

sft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
opt = torch.optim.AdamW(sft_model.parameters(), lr=LR_SFT)

train_loader = DataLoader(train_pref, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=sft_collate)
val_loader = DataLoader(val_pref, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=sft_collate)

In [8]:
print("===== SFT TRAINING =====")
for epoch in range(EPOCHS):
    sft_model.train()
    train_total = 0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = sft_model(**batch)
        loss = outputs.loss
        opt.zero_grad()
        loss.backward()
        opt.step()
        train_total += loss.item()
    train_loss = train_total / len(train_loader)
    
    sft_model.eval()
    val_total = 0
    with torch.no_grad():
        for step, batch in enumerate(val_loader):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = sft_model(**batch)
            loss = outputs.loss
            val_total += loss.item()
    val_loss = val_total / len(val_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

sft_model.save_pretrained(f"{OUT_DIR}/sft")

===== SFT TRAINING =====
Epoch 1/3 - Train Loss: 2.0173, Val Loss: 1.6261
Epoch 2/3 - Train Loss: 1.6266, Val Loss: 1.5125
Epoch 3/3 - Train Loss: 1.5110, Val Loss: 1.4488


In [9]:
# ================= REWARD MODEL =================
class RewardModel(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.head = nn.Linear(base.config.n_embd, 1)

    def forward(self, ids, attention_mask):
        h = self.base(input_ids=ids, attention_mask=attention_mask,
                      output_hidden_states=True).hidden_states[-1]
        pooled = masked_mean(h, attention_mask)
        return self.head(pooled).squeeze(-1)

def reward_collate(batch):
    def tok(p, r):
        t = tokenizer(p+r, truncation=True, max_length=MAX_LENGTH,
                      padding="max_length", return_tensors="pt")
        return t.input_ids.squeeze(0), t.attention_mask.squeeze(0)
    c_ids, c_mask, r_ids, r_mask = [], [], [], []
    for b in batch:
        ci, cm = tok(b["prompt"], b["chosen"])
        ri, rm = tok(b["prompt"], b["rejected"])
        c_ids.append(ci); c_mask.append(cm)
        r_ids.append(ri); r_mask.append(rm)
    return {
        "c_ids": torch.stack(c_ids),
        "c_mask": torch.stack(c_mask),
        "r_ids": torch.stack(r_ids),
        "r_mask": torch.stack(r_mask)
    }

reward_model = RewardModel(
    AutoModelForCausalLM.from_pretrained(f"{OUT_DIR}/sft")
).to(DEVICE)

opt = torch.optim.AdamW(reward_model.parameters(), lr=LR_REWARD)
loss_fn = nn.BCEWithLogitsLoss()

In [14]:
for batch in train_loader:
    print(batch)
    break

{'input_ids': tensor([[50256, 50256, 50256,  ...,     0, 50169,   224],
        [ 6090,   345,  4727,  ...,   389,  1541,  1762],
        [ 2061,   338,   534,  ...,   638,  1329,   278],
        [50256, 50256, 50256,  ...,    30, 30325,   232]]), 'attention_mask': tensor([[0, 0, 0,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1]]), 'labels': tensor([[50256, 50256, 50256,  ...,     0, 50169,   224],
        [ 6090,   345,  4727,  ...,   389,  1541,  1762],
        [ 2061,   338,   534,  ...,   638,  1329,   278],
        [50256, 50256, 50256,  ...,    30, 30325,   232]])}


In [ ]:
# ================= REWARD MODEL TRAINING WITH VALIDATION =================
optimizer = torch.optim.AdamW(reward_model.parameters(), lr=LR_REWARD)
loss_fn = nn.BCEWithLogitsLoss()

best_val_loss = float("inf")
reward_train_losses = []
reward_val_losses = []

for epoch in range(EPOCHS):
    # --- Training ---
    reward_model.train()
    train_loss_sum = 0
    for batch in train_loader:
        c_ids, c_mask = batch["chosen_ids"].to(DEVICE), batch["chosen_mask"].to(DEVICE)
        r_ids, r_mask = batch["rejected_ids"].to(DEVICE), batch["rejected_mask"].to(DEVICE)

        c_score = reward_model(c_ids, c_mask)
        r_score = reward_model(r_ids, r_mask)
        loss = loss_fn(c_score - r_score, torch.ones_like(c_score))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item()
    avg_train_loss = train_loss_sum / len(train_loader)
    reward_train_losses.append(avg_train_loss)

    # --- Validation ---
    reward_model.eval()
    val_loss_sum = 0
    with torch.no_grad():
        for batch in val_loader:
            c_ids, c_mask = batch["chosen_ids"].to(DEVICE), batch["chosen_mask"].to(DEVICE)
            r_ids, r_mask = batch["rejected_ids"].to(DEVICE), batch["rejected_mask"].to(DEVICE)

            c_score = reward_model(c_ids, c_mask)
            r_score = reward_model(r_ids, r_mask)
            loss = loss_fn(c_score - r_score, torch.ones_like(c_score))
            val_loss_sum += loss.item()

    avg_val_loss = val_loss_sum / len(val_loader)
    reward_val_losses.append(avg_val_loss)

    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(reward_model.state_dict(), f"{OUT_DIR}/best_reward_model.pt")

    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

KeyError: 'chosen'

In [ ]:
# ================= PPO =================
ppo_policy = AutoModelForCausalLM.from_pretrained(f"{OUT_DIR}/sft").to(DEVICE)
ppo_ref = AutoModelForCausalLM.from_pretrained(f"{OUT_DIR}/sft").to(DEVICE)
for p in ppo_ref.parameters():
    p.requires_grad = False

opt = torch.optim.AdamW(ppo_policy.parameters(), lr=LR_PPO)

In [ ]:
# ================= PPO =================
ppo_policy = AutoModelForCausalLM.from_pretrained(f"{OUT_DIR}/sft").to(DEVICE)
ppo_ref = AutoModelForCausalLM.from_pretrained(f"{OUT_DIR}/sft").to(DEVICE)
for p in ppo_ref.parameters():
    p.requires_grad = False

opt = torch.optim.AdamW(ppo_policy.parameters(), lr=LR_PPO)

print("===== PPO TRAINING =====")
ppo_train_losses = []
ppo_train_rewards = []
ppo_train_kls = []
ppo_val_rewards = []

for epoch in range(EPOCHS):
    # --- Training ---
    ppo_policy.train()
    for batch in train_loader:
        prompts = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        prompt_len = prompts.size(1)

        gen, gmask = generate(ppo_policy, prompts, mask)
        reward = reward_model(gen, gmask)

        logp = compute_logprob(ppo_policy, gen, gmask, prompt_len)
        logp_ref = compute_logprob(ppo_ref, gen, gmask, prompt_len)

        ratio = torch.exp(logp - logp_ref)
        clipped = torch.clamp(ratio, 1-PPO_CLIP, 1+PPO_CLIP)
        kl = logp - logp_ref
        loss = -torch.min(ratio*reward, clipped*reward).mean() + PPO_BETA*kl.mean()

        opt.zero_grad()
        loss.backward()
        opt.step()

        ppo_train_losses.append(loss.item())
        ppo_train_rewards.append(reward.mean().item())
        ppo_train_kls.append(kl.mean().item())

    # --- Validation ---
    ppo_policy.eval()
    val_reward_sum = 0
    with torch.no_grad():
        for batch in val_loader:
            prompts = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            prompt_len = prompts.size(1)

            gen, gmask = generate(ppo_policy, prompts, mask)
            reward = reward_model(gen, gmask)
            val_reward_sum += reward.mean().item()
    avg_val_reward = val_reward_sum / len(val_loader)
    ppo_val_rewards.append(avg_val_reward)

    print(f"Epoch {epoch+1}/{EPOCHS} - Avg Validation Reward: {avg_val_reward:.4f}")

ppo_policy.save_pretrained(f"{OUT_DIR}/ppo")

In [ ]:
# ================= DPO =================
dpo_policy = AutoModelForCausalLM.from_pretrained(f"{OUT_DIR}/sft").to(DEVICE)
dpo_ref = AutoModelForCausalLM.from_pretrained(f"{OUT_DIR}/sft").to(DEVICE)
for p in dpo_ref.parameters():
    p.requires_grad=False

opt = torch.optim.AdamW(dpo_policy.parameters(), lr=LR_DPO)

def get_logps(logits, labels, mask):
    logits = logits[:, :-1]
    labels = labels[:, 1:]
    mask = mask[:, 1:]
    logp = F.log_softmax(logits, dim=-1)
    tok = logp.gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    return (tok * mask).sum(-1)

In [ ]:
print("===== DPO TRAINING =====")
dpo_loss_log = []

for epoch in range(EPOCHS):
    for batch in DataLoader(train_pref, BATCH_SIZE, True):
        c_enc = tokenizer(batch["prompt"] + batch["chosen"], truncation=True, max_length=MAX_LENGTH,
                          padding="max_length", return_tensors="pt").to(DEVICE)
        r_enc = tokenizer(batch["prompt"] + batch["rejected"], truncation=True, max_length=MAX_LENGTH,
                          padding="max_length", return_tensors="pt").to(DEVICE)

        p_c = dpo_policy(input_ids=c_enc.input_ids, attention_mask=c_enc.attention_mask).logits
        p_r = dpo_policy(input_ids=r_enc.input_ids, attention_mask=r_enc.attention_mask).logits

        with torch.no_grad():
            r_c = dpo_ref(input_ids=c_enc.input_ids, attention_mask=c_enc.attention_mask).logits
            r_r = dpo_ref(input_ids=r_enc.input_ids, attention_mask=r_enc.attention_mask).logits

        loss = -F.logsigmoid(DPO_BETA*((get_logps(p_c,c_enc.input_ids,c_enc.attention_mask)
              - get_logps(p_r,r_enc.input_ids,r_enc.attention_mask))
              - (get_logps(r_c,c_enc.input_ids,c_enc.attention_mask)
                 - get_logps(r_r,r_enc.input_ids,r_enc.attention_mask)))).mean()

        opt.zero_grad()
        loss.backward()
        opt.step()
        dpo_loss_log.append(loss.item())

dpo_policy.save_pretrained(f"{OUT_DIR}/dpo")

In [ ]:
# ================= EVALUATION =================
def test_model(model, prompt):
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    out, _ = generate(model, enc.input_ids, enc.attention_mask)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text[len(prompt):].strip()

human_prompt = "What is the meaning of life?"
print("\n===== HUMAN PROMPT TEST =====")
print("Prompt:", human_prompt)
print("SFT Response:", test_model(sft_model, human_prompt))
print("PPO Response:", test_model(ppo_policy, human_prompt))
print("DPO Response:", test_model(dpo_policy, human_prompt))

In [ ]:
# ================= PLOTTING =================
plt.figure(figsize=(15,5))
plt.subplot(1,3,1)
plt.plot(ppo_train_losses)
plt.title("PPO Training Loss")
plt.xlabel("Step"); plt.ylabel("Loss")

plt.subplot(1,3,2)
plt.plot(ppo_val_rewards)
plt.title("PPO Rewards")
plt.xlabel("Step"); plt.ylabel("Reward")

plt.subplot(1,3,3)
plt.plot(dpo_loss_log)
plt.title("DPO Training Loss")
plt.xlabel("Step"); plt.ylabel("Loss")
plt.tight_layout()
plt.show()

In [ ]:
def compute_avg_reward(model, dataset, n_samples=32):
    total = 0
    for i, ex in enumerate(dataset.select(range(n_samples))):
        enc = tokenizer(ex["prompt"], return_tensors="pt").to(DEVICE)
        out, attn = generate(model, enc.input_ids, enc.attention_mask)
        total += reward_model(out, attn).mean().item()
    return total / n_samples

ppo_avg_reward = compute_avg_reward(ppo_policy, val_pref)
print(f"\nPPO Avg Reward on sample val set: {ppo_avg_reward:.4f}")